# NMEA 0183 — GPS dan GNSS

Penerima GPS, transponder AIS, dan instrumen kapal berbicara NMEA 0183: baris ASCII seperti `$GPGGA,...*47` dengan checksum XOR.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.8.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.8.0-preview.1"

## Urai sebuah kalimat

In [ ]:
using IoTCom.Net.Protocols.Nmea;
var gga = (GgaMessage)NmeaParser.Parse("$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M,46.9,M,,*47")!;
Console.WriteLine($"{gga.Latitude:0.0000}, {gga.Longitude:0.0000} · {gga.Satellites} satellites · {gga.AltitudeMeters} m");

## Penerima simulasi dan pembaca
`GnssState` menggabungkan GGA, RMC, GSA, dan GSV menjadi satu fix.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Transports;
var link = new InMemoryTransportListener();
var receiver = NmeaServer.Create(o => o.ListenInMemory(link));
await receiver.StartAsync();
var reader = NmeaReader.Create(o => o.UseInMemory(link));
await reader.ConnectAsync();
await Task.Delay(100);

var sim = new NmeaSimulator();
foreach (var s in sim.GenerateEpoch(DateTimeOffset.UtcNow, TimeSpan.FromSeconds(30))) await receiver.BroadcastAsync(s);
await Task.Delay(200);
var fix = reader.Gnss.Current;
Console.WriteLine($"{fix.Latitude:0.00000}, {fix.Longitude:0.00000} · {fix.SpeedKmh} km/h · {fix.SatellitesInView.Count} in view");

## Pemecahan masalah
- Tidak ada data di serial: kebanyakan penerima default 9600 atau 4800 baud, 8N1.
- Ada fix tetapi tanpa posisi: periksa status RMC `A` (aktif) vs `V` (void).

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*